In [ ]:
#import library
import pandas as pd
import numpy as np

#machine learning library
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

#wordcloud library
from wordcloud import WordCloud, STOPWORDS
import datetime, nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

#nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

# **Handling Imbalance Data**

In [ ]:
# human capital
human_cap = pd.read_csv("https://raw.githubusercontent.com/densaiko/data_science_learning/main/dataset/Human%20Capital.csv")
print("Diabetes data contain {} rows and {} columns \n".format(human_cap.shape[0], human_cap.shape[1]))
print(human_cap.info(), '\n')
human_cap.describe()

In [ ]:
def label_encoding(data):
  """
  Let's encode the non_numerical data into a numerical value
  data: your dataset
  """

  data_new = data.copy()
  data_new = data_new.dropna()

  #find non numerical variable/field
  object_data = data_new.select_dtypes(include=['object']).columns.to_list()

  #label encoder object
  label_encoding = LabelEncoder()

  #encode the data into a label
  for i in object_data:
    data_new[i] = label_encoding.fit_transform(data_new[i])

  return data_new

In [ ]:
new_human_cap = label_encoding(human_cap)
new_human_cap.head()

In [ ]:
X = new_human_cap.drop('is_promoted', axis=1)
y = new_human_cap['is_promoted']

#implement train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=43)

In [ ]:
def modelling(X_train, y_train, X_test, y_test):

  # modelling with logistic regression
  clf = GradientBoostingClassifier()
  clf.fit(X_train, y_train)

  # Evaluation
  y_predict_train = clf.predict(X_train)
  y_predict_test = clf.predict(X_test)

  training_acc = accuracy_score(y_train, y_predict_train)
  testing_acc = accuracy_score(y_test, y_predict_test)

  print("Training Accuracy: {}".format(training_acc))
  print("Testing Accuracy: {}".format(testing_acc))

In [ ]:
modelling(X_train, y_train, X_test, y_test)

## **Undersampling**

In [ ]:
#set the undersampling
undersample = RandomUnderSampler(sampling_strategy=0.5) #set your strategy

#fit the data
X_under, y_under = undersample.fit_resample(X_train, y_train)

print(Counter(y_train))
print(Counter(y_under))

In [ ]:
modelling(X_under, y_under, X_test, y_test)

## **Oversampling**

In [ ]:
#set the oversampling
oversample = RandomOverSampler(sampling_strategy=0.5) #set your strategy

#fit the data
X_over, y_over = oversample.fit_resample(X_train, y_train)

print(Counter(y_train))
print(Counter(y_over))

In [ ]:
modelling(X_over, y_over, X_test, y_test)

## **SMOTE**

In [ ]:
# Handling imbalance data
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=43, sampling_strategy=0.5)

# Fit the over sampling
X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train)

print("Before over sampling: {}".format(Counter(y_train)))
print("After over sampling: {}".format(Counter(y_train_smote)))

In [ ]:
modelling(X_train_smote, y_train_smote, X_test, y_test)



---



# **Text Pre-Processing**

In [ ]:
# news dataset
news_data = pd.read_csv("https://raw.githubusercontent.com/frfusch21/Data-Science-Modul/master/Datasets/News%20Title.csv",sep=";",header=0,index_col=0)
news_data.head()

In [ ]:
print("News data contain {} rows and {} columns \n".format(news_data.shape[0], news_data.shape[1]))
print(news_data.info(), '\n')
news_data.describe()

##**Data Cleansing**


###Removing Punctuations

In [ ]:
#list of punctuations
string.punctuation

In [ ]:
# example of taking out punctuations
nopunc = [char for char in news_data['News Title'][1] if char not in string.punctuation]
nopunc = ''.join(nopunc)
nopunc = nopunc.lower()

print("Before: {} \n After: {}".format(news_data['News Title'][1], nopunc))

###Tokenization


In [ ]:
# example tokenization
word_tokens = word_tokenize(nopunc)
word_tokens

###Removing Stopwords

In [ ]:
# example implementation of stopwords
stop_words = set(stopwords.words('english'))

clean_words = [w for w in word_tokens if not w in stop_words]
clean_words



---



In [ ]:
#apply to dataframe

def punctuation_removal(text):
  #remove punctuation that is not important
  nopunc = [char for char in text if char not in string.punctuation]
  nopunc = ''.join(nopunc)
  nopunc = nopunc.lower()
  return nopunc

def tokenization(text):
  word_tokens = word_tokenize(text)
  return word_tokens

def stopword_removal(text):
  #remove useless words in data
  stop_words = set(stopwords.words('english'))
  clean_words = [w for w in text if not w in stop_words]
  return clean_words


In [ ]:
news_data['News Title (Punctuation Removed)'] = news_data['News Title'].apply(punctuation_removal)
news_data['News Title (Tokenized)'] = news_data['News Title (Punctuation Removed)'].apply(tokenization)
news_data['News Title (Stopword Removed)'] = news_data['News Title (Tokenized)'].apply(stopword_removal)

In [ ]:
news_data.head()

#**Data Pre-processing**

###Stemming


In [ ]:
# example implementation of stemming
ps = PorterStemmer()

stemmed_words = []

for w in clean_words:
  stemmed_words.append(ps.stem(w))

' '.join(stemmed_words)


###Lemmatization

In [ ]:
# example implementation of lemmatization
lem = WordNetLemmatizer()

lem_words = []

for w in clean_words:
  lem_words.append(lem.lemmatize(w))

' '.join(lem_words)

In [ ]:
#apply to dataframe

def stemming(text):
  stemmed_words = []
  for w in text:
    stemmed_words.append(ps.stem(w))
  return ' '.join(stemmed_words)

def lemmatization(text):
  lem_words = []
  for w in text:
    lem_words.append(lem.lemmatize(w))
  return ' '.join(lem_words)


In [ ]:
news_data['News Title (Stemming)'] = news_data['News Title (Stopword Removed)'].apply(stemming)
news_data['News Title (Lemmatization)'] = news_data['News Title (Stopword Removed)'].apply(lemmatization)

news_data.head()

###Bag of Words

In [ ]:
#example Bag of Words

sent1 = 'my favorite search engine is google'
sent2 = 'hell ya google'

cv = CountVectorizer()

X = cv.fit_transform([sent1,sent2])

bow_sklearn = pd.DataFrame(X.toarray(),columns=cv.get_feature_names_out()) # newer version of scikit-learn use "cv.get_feature_names()"
bow_sklearn.head()

In [ ]:
# calculate the Bag of Word in news data
message_bow = CountVectorizer().fit_transform(np.array(list(news_data['News Title (Lemmatization)'])))
message_bow.shape

In [ ]:
#split the dataset into 80% training data and 20% testing data
X_train_bow, X_test_bow, y_train_bow, y_test_bow = train_test_split(message_bow, news_data['Category'], test_size=0.20, random_state=0)


In [ ]:
# naive-bayes model
def modelling(model, X_train, y_train, X_text, y_test):
  #basic machine learning
  MNB = model
  MNB.fit(X_train, y_train)

  y_train_pred = MNB.predict(X_train)
  y_test_pred = MNB.predict(X_text)

  # Compute accuracy on training set
  train_accuracy = MNB.score(X_train, y_train)

  # Compute accuracy on testing set
  test_accuracy = MNB.score(X_text, y_test)

  print('Training accuracy {:.2f}%'.format(train_accuracy*100))
  print('Testing accuracy {:.2f}%'.format(test_accuracy*100))

In [ ]:
# modelling and test the model with the test data
modelling(MultinomialNB(), X_train_bow, y_train_bow, X_test_bow, y_test_bow)

###TF-IDF


In [ ]:
idf_matrix = TfidfVectorizer(min_df=0., max_df=1., use_idf=True).fit_transform(np.array(list(news_data['News Title (Lemmatization)'])))
idf_matrix.shape

In [ ]:
#split the dataset into 80% training data and 20% testing data
X_train_idf, X_test_idf, y_train_idf, y_test_idf = train_test_split(idf_matrix, news_data['Category'], test_size=0.20, random_state=0)

In [ ]:
# modelling and test the model with the test data
modelling(MultinomialNB(), X_train_idf, y_train_idf, X_test_idf, y_test_idf)

## **Logistic Regression Model with TF-IDF**


In [ ]:
def modelling2(model, X_train, y_train, X_test, y_test):
    lr_model = model
    lr_model.fit(X_train, y_train)

    y_train_pred = lr_model.predict(X_train)
    y_test_pred = lr_model.predict(X_test)

    # Compute accuracy on training set
    train_accuracy = lr_model.score(X_train, y_train)

    # Compute accuracy on testing set
    test_accuracy = lr_model.score(X_test, y_test)

    print('Training accuracy: {:.2f}%'.format(train_accuracy * 100))
    print('Testing accuracy: {:.2f}%'.format(test_accuracy * 100))

In [ ]:
# modelling and test the model with the test data
modelling2(LogisticRegression(max_iter=1000), X_train_idf, y_train_idf, X_test_idf, y_test_idf)